# **Variance reduction techniques**

In this notebook we price an European call option with Monte Carlo method, comparing different variance reduction techniques.

In [ ]:
import numpy as np
from scipy.stats import norm

Assume Black-Scholes framework.

In [ ]:
def asset_price(S, sigma, r, T, Z):
  return S*np.exp((r-0.5*sigma**2)*T+sigma*np.sqrt(T)*Z)

In [ ]:
def bs_formula(S_0, K, sigma, r, T, option_type="call"):
  d1=(np.log(S_0/K)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
  d2=d1-sigma*np.sqrt(T)
  if option_type=="call":
    price=S_0*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
  elif option_type=="put":
    price=K*np.exp(-r*T)*norm.cdf(-d2)-S_0*norm.cdf(-d1)
  else:
    raise ValueError ("option_type should be 'call' or 'put'")

  return price

## Classic Monte Carlo

In [ ]:
def mc(S_0, K, sigma, r, T, N, option_type="call"):
  Z=np.random.standard_normal(N)
  S_T=asset_price(S_0, sigma, r, T, Z)

  if option_type=="call":
    payoff=np.maximum(S_T-K,0)
  elif option_type=="put":
    payoff=np.maximum(K-S_T,0)
  else:
    raise ValueError("option_type should be 'call' or 'put'")
  discounted_payoff=np.exp(-r*T)*payoff

  price=np.mean(discounted_payoff)
  std_error=np.std(discounted_payoff, ddof=1)/np.sqrt(N)
  ci_low=price-1.96*std_error
  ci_high=price+1.96*std_error

  return price, std_error, (ci_low, ci_high)

In [ ]:
S_0=100
K=100
sigma=0.2
r=0.02
T=2
N=100000

In [ ]:
price=mc(S_0, K, sigma, r, T, N, option_type="call")
print("### CLASSIC MONTE CARLO ###")
print("Option price:", price[0])
print("Standard error:", price[1])
print("Confidence Interval (95%):", price[2])

### CLASSIC MONTE CARLO ###
Option price: 13.066559641104547
Standard error: 0.06604875853102836
Confidence Interval (95%): (np.float64(12.937104074383731), np.float64(13.196015207825363))


## Antithetic variables

Use pairs of opposite normal shocks to reduce the variance of the Monte Carlo estimator.

In [ ]:
def mc_anthitetic_variable(S_0, K, sigma, r, T, N, option_type="call"):
  Z=np.random.standard_normal(N//2)

  S_p = asset_price(S_0, sigma, r, T, Z)
  S_m = asset_price(S_0, sigma, r, T, -Z)

  if option_type == "call":
      payoff_p = np.maximum(S_p - K, 0)
      payoff_m = np.maximum(S_m - K, 0)
  elif option_type == "put":
      payoff_p = np.maximum(K - S_p, 0)
      payoff_m = np.maximum(K - S_m, 0)
  else:
      raise ValueError("option_type should be 'call' or 'put'")

  payoff = (payoff_p + payoff_m) / 2
  discounted_payoff = np.exp(-r * T) * payoff

  price = np.mean(discounted_payoff)
  std_error = np.std(discounted_payoff, ddof=1) / np.sqrt(N // 2)
  ci_low = price - 1.96 * std_error
  ci_high = price + 1.96 * std_error

  return price, std_error, (ci_low, ci_high)


In [ ]:
price_av=mc_anthitetic_variable(S_0, K, sigma, r, T, N, option_type="call")
print("### ANTITHETIC VARIABLE TECHNIQUE ###")
print("Option price:", price_av[0])
print("Standard error:", price_av[1])
print("Confidence Interval (95%):", price_av[2])

### ANTITHETIC VARIABLE TECHNIQUE ###
Option price: 13.160624811873745
Standard error: 0.05133282417502883
Confidence Interval (95%): (np.float64(13.060012476490689), np.float64(13.261237147256802))


## Control variates

Exploit a related derivative with known closed-form price to improve the efficiency of the estimator.



In [ ]:
def mc_control_variate(S_0, K, sigmaA, sigmaB, r, T, N, option_type="call"):
  bs_priceB=bs_formula(S_0, K, sigmaB, r, T, option_type)
  Z=np.random.standard_normal(N)

  SA_T=asset_price(S_0, sigmaA, r, T, Z)
  SB_T=asset_price(S_0, sigmaB, r, T, Z)

  if option_type=="call":
    payoffA=np.maximum(SA_T-K,0)
    payoffB=np.maximum(SB_T-K,0)
  elif option_type=="put":
    payoffA=np.maximum(K-SA_T,0)
    payoffB=np.maximum(K-SB_T,0)
  else:
    raise ValueError("option_type should be 'call' or 'put'")
  discounted_payoffA=np.exp(-r*T)*payoffA
  discounted_payoffB=np.exp(-r*T)*payoffB

  sample=discounted_payoffA+bs_priceB-discounted_payoffB
  price=np.mean(sample)
  std_error = np.std(sample, ddof=1) / np.sqrt(N)
  ci_low = price - 1.96 * std_error
  ci_high = price + 1.96 * std_error

  return price, std_error, (ci_low, ci_high)

In [ ]:
sigma_B=0.1

price_cv=mc_control_variate(S_0, K, sigma, sigma_B, r, T, N, option_type="call")
print("### ANTITHETIC VARIABLE TECHNIQUE ###")
print("Option price:", price_cv[0])
print("Standard error:", price_cv[1])
print("Confidence Interval (95%):", price_cv[2])

### ANTITHETIC VARIABLE TECHNIQUE ###
Option price: 13.038012416663742
Standard error: 0.03409356416078024
Confidence Interval (95%): (np.float64(12.971189030908613), np.float64(13.10483580241887))


## Importance sampling

Shift the sampling distribution toward more relevant outcomes and correct the estimator with a likelihood ratio.

In [ ]:
def mc_importance_sampling(S_0, K, sigma, mu, r, T, N, option_type="call"):
    Z_tilde = np.random.standard_normal(N) + mu
    S_T = asset_price(S_0, sigma, r, T, Z_tilde)

    if option_type == "call":
        payoff = np.maximum(S_T - K, 0)
    elif option_type == "put":
        payoff = np.maximum(K - S_T, 0)
    else:
        raise ValueError("option_type should be 'call' or 'put'")

    plus_term=np.exp(-mu * Z_tilde + 0.5 * mu**2)
    discounted_payoff=np.exp(-r * T)*payoff*plus_term

    price=np.mean(discounted_payoff)
    std_error=np.std(discounted_payoff, ddof=1) / np.sqrt(N)
    ci_low=price-1.96*std_error
    ci_high=price+1.96*std_error

    return price, std_error, (ci_low, ci_high)

In [ ]:
mu=0.2

price_im=mc_importance_sampling(S_0, K, sigma, mu, r, T, N, option_type="call")
print("### IMPORTANCE SAMPLING TECHNIQUE ###")
print("Option price:", price_im[0])
print("Standard error:", price_im[1])
print("Confidence Interval (95%):", price_im[2])

### IMPORTANCE SAMPLING TECHNIQUE ###
Option price: 13.08225120871132
Standard error: 0.051809784022925826
Confidence Interval (95%): (np.float64(12.980704032026384), np.float64(13.183798385396255))


## Comparision

In [ ]:
print("Standard Error Classic Monte Carlo:", price[1])
print("Standard Error Antithetic Variable:", price_av[1])
print("Standard Error Control Variate:", price_cv[1])
print("Standard Error Importance Sampling:", price_im[1])

Standard Error Classic Monte Carlo: 0.06604875853102836
Standard Error Antithetic Variable: 0.05133282417502883
Standard Error Control Variate: 0.03409356416078024
Standard Error Importance Sampling: 0.051809784022925826
